In [ ]:
# Install required packages if not already installed
# !pip install fastmcp requests python-dotenv

import requests
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# First, let's generate a token from our server (you'll run this after starting the server)
def get_test_token():
    """
    In a real application, you would get this token from your authentication service.
    For this example, the server will print the token when it starts.
    """
    # This is the token that will be printed when you run the server
    # Copy it from the server output and paste it here
    return "YOUR_TOKEN_HERE"  # Replace with actual token from server output

# Test authentication with proper token
def test_authenticated_server():
    url = "http://localhost:8001/mcp"
    
    # Get the token (in production, this would come from your auth service)
    token = get_test_token()
    
    if token == "YOUR_TOKEN_HERE":
        print("⚠️  Please start the server first and copy the generated token!")
        print("Run the server with: python test.py")
        print("Then copy the token from the server output and update the get_test_token() function")
        return
    
    # Use Bearer token authentication (the proper way)
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    
    # Test the auth_bearer tool
    tool_payload = {
        "jsonrpc": "2.0",
        "id": "test-1", 
        "method": "tools/call",
        "params": {
            "name": "auth_bearer",
            "arguments": {}
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=tool_payload)
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")
        
        if response.status_code == 200:
            result = response.json()
            print("✅ Authentication successful!")
            print(f"Result: {result}")
        else:
            print("❌ Authentication failed!")
            
    except Exception as e:
        print(f"Error: {e}")

# Run the test
test_authenticated_server()

In [37]:
def example_client_code():
    """
    THIS IS HOW YOU SEND THE HEADERS FROM YOUR CLIENT:
    """
    import requests
    
    url = "http://localhost:8001/mcp"
    
    # THE KEY: Send user_id and chatbot_id as custom headers
    headers = {
        "Content-Type": "application/json",
        "X-User-ID": "your_user_123",        # ← YOUR USER ID HERE
        "X-Chatbot-ID": "your_chatbot_456"   # ← YOUR CHATBOT ID HERE
    }
    
    payload = {
        "jsonrpc": "2.0",
        "id": "test",
        "method": "tools/call",
        "params": {
            "name": "extract_user_and_chatbot_ids",
            "arguments": {}
        }
    }
    
    # Send the request
    response = requests.post(url, headers=headers, json=payload)
    return response

print(example_client_code())

<Response [406]>


In [ ]:
# Test with the programmatically generated token
def test_with_generated_token():
    url = "http://localhost:8001/mcp"
    
    # Use the token from Cell 1
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    # Test calling the auth_bearer tool
    payload = {
        "jsonrpc": "2.0",
        "id": "test-auth",
        "method": "tools/call", 
        "params": {
            "name": "auth_bearer",
            "arguments": {}
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        print(f"Status: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            print("✅ Success!")
            print(json.dumps(result, indent=2))
        else:
            print("❌ Failed!")
            print(f"Response: {response.text}")
            
    except requests.exceptions.ConnectionError:
        print("❌ Connection failed. Make sure the server is running on port 8001")
    except Exception as e:
        print(f"Error: {e}")

# Run the test
test_with_generated_token()

In [ ]:
# Using FastMCP Client with authentication
from fastmcp import Client

async def test_with_fastmcp_client():
    """Test using the FastMCP Client with proper authentication"""
    
    config = {
        "mcpServers": {
            "auth_server": {
                "url": "http://localhost:8001/mcp",
                "transport": "streamable_http",
                "headers": {
                    "Authorization": f"Bearer {token}"
                }
            }
        }
    }
    
    try:
        client = Client(config)
        
        async with client:
            # List available tools
            tools = await client.list_tools()
            print("Available tools:")
            for tool in tools:
                print(f"- {tool.name}: {tool.description}")
            
            # Call the auth_bearer tool
            result = await client.call_tool("auth_bearer", {})
            print(f"\nTool result: {result}")
            
    except Exception as e:
        print(f"Error with FastMCP client: {e}")

# Run the async function
import asyncio
await test_with_fastmcp_client()

In [ ]:
# Decode and inspect the token to see what information it contains
import base64
import json

def decode_jwt_payload(token):
    """Decode JWT payload to see the claims"""
    try:
        # JWT has three parts separated by dots
        parts = token.split('.')
        if len(parts) != 3:
            return "Invalid JWT format"
        
        # Decode the payload (second part)
        payload = parts[1]
        
        # Add padding if needed
        payload += '=' * (4 - len(payload) % 4)
        
        # Decode base64
        decoded = base64.urlsafe_b64decode(payload)
        
        # Parse JSON
        claims = json.loads(decoded)
        
        return claims
        
    except Exception as e:
        return f"Error decoding token: {e}"

# Decode our test token
claims = decode_jwt_payload(token)
print("Token Claims:")
print(json.dumps(claims, indent=2))

print(f"\nUser ID: {claims.get('sub')}")
print(f"Chatbot ID: {claims.get('chatbot_id')}")
print(f"Scopes: {claims.get('scope', [])}")
print(f"Expires: {claims.get('exp')}")

In [ ]:
# Complete working example: Start to finish
print("🚀 Complete Authentication Example")
print("="*50)

print("\n1. ✅ Generated Token:")
print(f"   User: {user_id}")
print(f"   Chatbot: {chatbot_id}")

print("\n2. 🔍 Token Claims:")
for key, value in claims.items():
    print(f"   {key}: {value}")

print("\n3. 🌐 Making Authenticated Request...")

# Make the request
url = "http://localhost:8001/mcp"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
payload = {
    "jsonrpc": "2.0",
    "id": "final-test",
    "method": "tools/call",
    "params": {"name": "auth_bearer", "arguments": {}}
}

try:
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        result = response.json()
        print("   ✅ SUCCESS!")
        print(f"   Server Response: {result}")
    else:
        print(f"   ❌ Failed with status {response.status_code}")
        print(f"   Response: {response.text}")
        
except requests.exceptions.ConnectionError:
    print("   ❌ Server not running. Start it with: python test.py")
except Exception as e:
    print(f"   ❌ Error: {e}")

print("\n" + "="*50)
print("🎯 Key Points:")
print("- Use Bearer token authentication, not custom headers")
print("- Token contains user_id in 'sub' claim")
print("- Additional data (like chatbot_id) goes in custom claims")
print("- Server validates token and extracts user information")
print("- This is much more secure than passing raw headers")
